# Marine Spatial Planning Violation Signals

Colab/Jupyter notebook using Google Earth Engine and `geemap` to flag changes to the sea area that may indicate marine spatial planning violations.

The analysis is limited to **baseline water**: pixels classified as water by NDWI in the base period at the start of the date range (e.g. 2021 for a 2021-2025 run). Everything afterwards measures what happened to that water area, such as reclamation or construction. Land that was already land in the base period is ignored.

The outputs are **review signals, not legal conclusions**. Every flagged zone needs checking by a planner, regulator, or domain expert.

Cells whose first line is `# lib-cell: ...` contain only imports, constants, and function definitions. The tests in `scripts/` load and check exactly those cells, so keep side effects (map display, `getInfo()` calls) out of them.

## 1. Install and import packages

In Colab, run the install cell first. Locally, run `pip install -r requirements.txt` once and skip it.

In [ ]:
%pip install -q earthengine-api geemap geopandas pandas python-dotenv shapely matplotlib

In [1]:
# lib-cell: imports
import calendar
import datetime as dt

import ee
import pandas as pd

In [2]:
import geemap
import matplotlib.pyplot as plt

In [3]:
import os

try:
    from dotenv import load_dotenv

    load_dotenv()  # picks up .env next to the notebook when run locally
except ImportError:
    pass

# Google Cloud project registered for Earth Engine. Set EE_PROJECT in .env (see .env.example),
# or type it here when running in Colab, e.g. EE_PROJECT = "my-ee-project".
EE_PROJECT = os.environ.get("EE_PROJECT", "")

try:
    ee.Initialize(project=EE_PROJECT or None)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT or None)

## 2. Analysis parameters

The base period runs from `START_DATE` for `BASELINE_MONTHS`. A full year smooths out tides and seasonal turbidity, so the baseline water mask is more stable than one built from a single 3- or 6-month window.

Thresholds are starting points. Tune them per site: turbid water and tides both shift NDWI.

In [4]:
START_DATE = "2021-01-01"
END_DATE = "2025-01-01"
INTERVAL_MONTHS = 6  # use 3 or 6
BASELINE_MONTHS = 12  # base period length from START_DATE; baseline water is mapped from this composite
CLOUD_PROBABILITY_THRESHOLD = 40
NDWI_THRESHOLD = 0.0  # NDWI above this is water

# Concern-zone rules
NDBI_GAIN_THRESHOLD = 0.1  # lost water whose NDBI rose by more than this is marked builtup_gain
REEF_BUFFER_M = 500  # flagged change within this distance of a mapped reef is marked near_reef
MIN_ZONE_AREA_HA = 0.5  # drop concern zones smaller than this

ANALYSIS_SCALE = 10  # metres; raise to 20-30 for large areas if Earth Engine times out

## 3. Draw or load the coverage area

Draw a polygon on the map using the left toolbar, then run the next cell to capture it as `aoi`. If nothing is drawn, the notebook uses the fallback polygon (Jakarta Bay). Users elsewhere can replace `FALLBACK_COORDS` or the map center.

In [5]:
MAP_CENTER = [-6.2, 106.8]  # lat, lon
FALLBACK_COORDS = [
    [106.70, -6.10],
    [106.90, -6.10],
    [106.90, -6.30],
    [106.70, -6.30],
    [106.70, -6.10],
]

Map = geemap.Map(center=MAP_CENTER, zoom=10)
Map.add_basemap("SATELLITE")
Map

Map(center=[-6.2, 106.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [6]:
aoi = Map.user_roi
if aoi is None:
    aoi = ee.Geometry.Polygon([FALLBACK_COORDS])
    print("No drawn AOI found. Using fallback example polygon.")
else:
    print("Using drawn AOI from geemap.")

Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map.centerObject(aoi, 11)

Using drawn AOI from geemap.


## 4. Helper functions

In [7]:
# lib-cell: date windows
def add_months(date, months):
    """Shift a date by whole months, clamping the day to the target month's length."""
    month_index = date.month - 1 + months
    year = date.year + month_index // 12
    month = month_index % 12 + 1
    day = min(date.day, calendar.monthrange(year, month)[1])
    return dt.date(year, month, day)


def date_windows(start_date, end_date, interval_months=3):
    """Split [start_date, end_date) into consecutive windows of interval_months.

    Every boundary is measured from start_date, so clamping (e.g. Jan 31 -> Feb 28)
    does not drift into later windows. The last window is cut off at end_date.
    """
    start = dt.date.fromisoformat(start_date)
    end = dt.date.fromisoformat(end_date)
    if interval_months < 1:
        raise ValueError("interval_months must be at least 1")
    if start >= end:
        raise ValueError("start_date must be before end_date")

    windows = []
    current = start
    step = 0
    while current < end:
        step += 1
        next_date = min(add_months(start, step * interval_months), end)
        windows.append((current.isoformat(), next_date.isoformat()))
        current = next_date

    return windows

In [8]:
# lib-cell: Sentinel-2 composites
INDEX_NAMES = ["NDVI", "NDWI", "NDBI"]
CLOUDY_SCL_CLASSES = [3, 8, 9, 10, 11]  # cloud shadow, cloud medium/high probability, cirrus, snow


def sentinel2_collection(aoi, start_date, end_date, cloud_probability_threshold=40):
    """Cloud-masked Sentinel-2 SR images (reflectance 0-1) over aoi between the dates."""
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(aoi).filterDate(start_date, end_date)
    clouds = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(aoi).filterDate(start_date, end_date)

    joined = ee.ImageCollection(
        ee.Join.saveFirst("cloud_mask").apply(
            primary=s2,
            secondary=clouds,
            condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
        )
    )

    def mask_clouds(image):
        cloud_prob = ee.Image(image.get("cloud_mask")).select("probability")
        clear_scl = image.select("SCL").remap(CLOUDY_SCL_CLASSES, [0] * len(CLOUDY_SCL_CLASSES), 1)
        clear = cloud_prob.lt(cloud_probability_threshold).And(clear_scl)
        return image.select("B.*").updateMask(clear).divide(10000).copyProperties(image, ["system:time_start"])

    return joined.map(mask_clouds)


def window_image_counts(aoi, windows, cloud_probability_threshold=40):
    """Number of Sentinel-2 images in each (start, end) window, fetched in one request."""
    counts = ee.List(
        [sentinel2_collection(aoi, start, end, cloud_probability_threshold).size() for start, end in windows]
    )
    return counts.getInfo()


def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndbi = image.normalizedDifference(["B11", "B8"]).rename("NDBI")
    return image.addBands([ndvi, ndwi, ndbi])


def make_composite(aoi, start_date, end_date, cloud_probability_threshold=40):
    """Median composite with NDVI/NDWI/NDBI bands. The window must contain at least one image."""
    collection = sentinel2_collection(aoi, start_date, end_date, cloud_probability_threshold)
    return add_indices(collection.median().clip(aoi)).set(
        {"window_start": start_date, "window_end": end_date, "image_count": collection.size()}
    )

In [9]:
# lib-cell: baseline water and change
def water_mask(image, ndwi_threshold=0.0):
    """0/1 NDWI water mask; masked where the image is cloud-masked."""
    return image.select("NDWI").gt(ndwi_threshold).rename("water")


def baseline_water(baseline_image, ndwi_threshold=0.0):
    """0/1 water mask of the base period. Cloud gaps count as 0, so they are left out of the analysis."""
    return water_mask(baseline_image, ndwi_threshold).unmask(0).rename("baseline_water")


def water_loss(baseline, image, ndwi_threshold=0.0):
    """1 where baseline water is no longer water in image, 0 where it still is.

    Masked outside baseline water and where image is cloud-masked.
    """
    return water_mask(image, ndwi_threshold).Not().updateMask(baseline).rename("water_to_land")


def index_change(start_image, end_image):
    """End minus start for each index, as bands NDVI_change, NDWI_change, NDBI_change."""
    return (
        end_image.select(INDEX_NAMES)
        .subtract(start_image.select(INDEX_NAMES))
        .rename([f"{name}_change" for name in INDEX_NAMES])
    )


def area_ha(mask_image, region, scale=10):
    """ee.Dictionary of hectares covered by each 0/1 band of mask_image within region."""
    return mask_image.multiply(ee.Image.pixelArea().divide(10000)).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, scale=scale, maxPixels=1e13
    )


def features_to_frame(feature_collection_info):
    """DataFrame of feature properties from a FeatureCollection getInfo() result."""
    return pd.DataFrame([feature["properties"] for feature in feature_collection_info["features"]])

In [10]:
# lib-cell: time-series summary
def window_stats(image, baseline, aoi, ndwi_threshold=0.0, scale=10):
    """ee.Feature describing one composite inside the baseline water area.

    clear_ha is baseline water visible (not cloud-masked) in this window; water_ha and
    lost_ha split it into still-water and no-longer-water. lost_fraction = lost_ha / clear_ha,
    so it is not biased by clouds. Index means are taken over clear baseline water only.
    """
    in_baseline = image.updateMask(baseline)
    means = in_baseline.select(INDEX_NAMES).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=scale, maxPixels=1e13
    )
    lost = water_loss(baseline, image, ndwi_threshold)
    areas = area_ha(
        ee.Image.cat(
            [
                in_baseline.select("NDWI").mask().gt(0).And(baseline).rename("clear_ha"),
                lost.Not().rename("water_ha"),
                lost.rename("lost_ha"),
            ]
        ),
        aoi,
        scale,
    )
    baseline_ha = ee.Number(area_ha(baseline, aoi, scale).get("baseline_water")).max(1e-9)
    clear_ha = ee.Number(areas.get("clear_ha"))
    return ee.Feature(
        None,
        means.combine(areas)
        .set("baseline_water_ha", baseline_ha)
        .set("clear_fraction", clear_ha.divide(baseline_ha))
        .set("lost_fraction", ee.Number(areas.get("lost_ha")).divide(clear_ha.max(1e-9)))
        .set("window_start", image.get("window_start"))
        .set("window_end", image.get("window_end"))
        .set("image_count", image.get("image_count")),
    )


def time_series_table(composites, baseline, aoi, ndwi_threshold=0.0, scale=10):
    """One row per composite, fetched in a single request and sorted by window_start."""
    stats = ee.FeatureCollection(
        [window_stats(image, baseline, aoi, ndwi_threshold, scale) for image in composites]
    )
    columns = [
        "window_start",
        "window_end",
        "image_count",
        "clear_fraction",
        "water_ha",
        "lost_ha",
        "lost_fraction",
        *INDEX_NAMES,
        "baseline_water_ha",
    ]
    return (
        features_to_frame(stats.getInfo())
        .reindex(columns=columns)
        .sort_values("window_start")
        .reset_index(drop=True)
    )

In [11]:
# lib-cell: Allen Coral Atlas
CORAL_ATLAS_ASSET = "ACA/reef_habitat/v2_0"
GEOMORPHIC_CLASSES = {
    11: "Shallow Lagoon",
    12: "Deep Lagoon",
    13: "Inner Reef Flat",
    14: "Outer Reef Flat",
    15: "Reef Crest",
    16: "Terrestrial Reef Flat",
    21: "Sheltered Reef Slope",
    22: "Reef Slope",
    23: "Plateau",
    24: "Back Reef Slope",
    25: "Patch Reef",
}
BENTHIC_CLASSES = {
    11: "Sand",
    12: "Rubble",
    13: "Rock",
    14: "Seagrass",
    15: "Coral/Algae",
    18: "Microalgal Mats",
}


def coral_atlas():
    """Allen Coral Atlas image with bands geomorphic, benthic, reef_mask (0 = unmapped / not reef)."""
    return ee.Image(CORAL_ATLAS_ASSET)


def class_area_groups(class_image, region, scale=10):
    """Hectares per non-zero class value of a single-band image, as a list of {"class", "sum"} dicts."""
    result = (
        ee.Image.pixelArea()
        .divide(10000)
        .addBands(class_image.selfMask())
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
            geometry=region,
            scale=scale,
            maxPixels=1e13,
        )
    )
    return result.get("groups")


def class_area_table(groups, class_names):
    """DataFrame of class name and hectares, largest first, from class_area_groups(...).getInfo()."""
    rows = [
        {"class": group["class"], "name": class_names.get(group["class"], "Unknown"), "area_ha": group["sum"]}
        for group in groups
    ]
    frame = pd.DataFrame(rows, columns=["class", "name", "area_ha"])
    return frame.sort_values("area_ha", ascending=False).reset_index(drop=True)


def near_reef_mask(buffer_m=500):
    """0/1 image: 1 within buffer_m metres of an Allen Coral Atlas reef pixel."""
    reef = coral_atlas().select("reef_mask").unmask(0).eq(1)
    distance = reef.distance(ee.Kernel.euclidean(buffer_m, "meters"))
    return distance.lte(buffer_m).unmask(0).rename("near_reef")

In [12]:
# lib-cell: concern flags and zones
FLAG_BANDS = ["water_to_land", "builtup_gain"]


def concern_flags(baseline, baseline_image, end_image, ndwi_threshold=0.0, ndbi_gain_threshold=0.1):
    """0/1 bands water_to_land and builtup_gain, defined only inside baseline water.

    water_to_land: baseline water that is no longer water at the end (possible reclamation).
    builtup_gain: the part of it whose NDBI rose by more than ndbi_gain_threshold since the
    baseline, i.e. it looks built-up rather than exposed mud or sand fill.
    """
    lost = water_loss(baseline, end_image, ndwi_threshold).unmask(0).And(baseline)
    ndbi_gain = index_change(baseline_image, end_image).select("NDBI_change").gt(ndbi_gain_threshold)
    return ee.Image.cat([lost.rename("water_to_land"), lost.And(ndbi_gain).rename("builtup_gain")])


def concern_zones(flags, aoi, near_reef=None, scale=10, min_area_ha=0.5, max_zones=500):
    """Polygons of connected flagged pixels, with hectares of each signal inside them.

    Each zone gets zone_ha, <flag>_ha for every flag band, near_reef_ha (when near_reef
    is given), and centroid lon/lat. Zones are sorted largest first.
    """
    any_flag = flags.reduce(ee.Reducer.max()).rename("any_flag").toInt()
    zones = any_flag.selfMask().reduceToVectors(
        geometry=aoi,
        scale=scale,
        geometryType="polygon",
        eightConnected=True,
        labelProperty="any_flag",
        maxPixels=1e13,
    )

    measured = any_flag.rename("zone").addBands(flags)
    if near_reef is not None:
        measured = measured.addBands(any_flag.And(near_reef).rename("near_reef"))
    hectares = measured.unmask(0).multiply(ee.Image.pixelArea().divide(10000))
    hectares = hectares.rename(measured.bandNames().map(lambda name: ee.String(name).cat("_ha")))

    def add_centroid(zone):
        coords = zone.geometry().centroid(1).coordinates()
        return zone.set({"lon": coords.get(0), "lat": coords.get(1)})

    return (
        hectares.reduceRegions(collection=zones, reducer=ee.Reducer.sum(), scale=scale)
        .filter(ee.Filter.gte("zone_ha", min_area_ha))
        .sort("zone_ha", False)
        .limit(max_zones)
        .map(add_centroid)
    )


def zone_table(zones_info, min_signal_ha=0.0):
    """DataFrame of concern zones from concern_zones(...).getInfo(), with signal summary columns."""
    frame = features_to_frame(zones_info)
    signal_columns = [f"{band}_ha" for band in FLAG_BANDS]
    if frame.empty:
        return pd.DataFrame(columns=["zone_id", "zone_ha", *signal_columns, "signals", "signal_count", "lon", "lat"])

    present = frame[signal_columns].gt(min_signal_ha)
    frame["signals"] = present.apply(
        lambda row: ", ".join(band for band, column in zip(FLAG_BANDS, signal_columns) if row[column]), axis=1
    )
    frame["signal_count"] = present.sum(axis=1)
    frame["zone_id"] = range(1, len(frame) + 1)

    columns = ["zone_id", "zone_ha", *signal_columns]
    if "near_reef_ha" in frame:
        columns.append("near_reef_ha")
    columns += ["signals", "signal_count", "lon", "lat"]
    return frame[columns].sort_values(["signal_count", "zone_ha"], ascending=False).reset_index(drop=True)

## 5. Baseline water and the time series

The baseline composite covers `START_DATE` to `START_DATE + BASELINE_MONTHS`. Pixels with NDWI above `NDWI_THRESHOLD` in it form the baseline water area that every later step measures against.

Windows with no Sentinel-2 images are skipped.

In [13]:
baseline_start = START_DATE
baseline_end = min(add_months(dt.date.fromisoformat(START_DATE), BASELINE_MONTHS), dt.date.fromisoformat(END_DATE)).isoformat()
if window_image_counts(aoi, [(baseline_start, baseline_end)], CLOUD_PROBABILITY_THRESHOLD) == [0]:
    raise RuntimeError("No Sentinel-2 images in the base period. Move START_DATE or lengthen BASELINE_MONTHS.")

baseline_image = make_composite(aoi, baseline_start, baseline_end, CLOUD_PROBABILITY_THRESHOLD)
baseline = baseline_water(baseline_image, NDWI_THRESHOLD)
baseline_water_ha = area_ha(baseline, aoi, ANALYSIS_SCALE).getInfo()["baseline_water"]
print(f"Base period {baseline_start} to {baseline_end}: {baseline_water_ha:,.1f} ha of baseline water")

Base period 2021-01-01 to 2022-01-01: 11,622.4 ha of baseline water


In [14]:
windows = date_windows(START_DATE, END_DATE, INTERVAL_MONTHS)
counts = window_image_counts(aoi, windows, CLOUD_PROBABILITY_THRESHOLD)

window_table = pd.DataFrame(windows, columns=["window_start", "window_end"]).assign(sentinel2_image_count=counts)
valid_windows = [window for window, count in zip(windows, counts) if count > 0]
skipped = len(windows) - len(valid_windows)
if skipped:
    print(f"Skipping {skipped} window(s) with no Sentinel-2 images.")
if not valid_windows:
    raise RuntimeError("No window has Sentinel-2 images. Widen the dates or the interval.")

composites = [make_composite(aoi, start, end, CLOUD_PROBABILITY_THRESHOLD) for start, end in valid_windows]
end_image = composites[-1]
window_table

,window_start,window_end,sentinel2_image_count
0,2021-01-01,2021-07-01,72
1,2021-07-01,2022-01-01,74
2,2022-01-01,2022-07-01,70
3,2022-07-01,2023-01-01,74
4,2023-01-01,2023-07-01,72
5,2023-07-01,2024-01-01,72
6,2024-01-01,2024-07-01,74
7,2024-07-01,2025-01-01,78


In [15]:
baseline_map = geemap.Map()
baseline_map.centerObject(aoi, 11)
baseline_map.add_basemap("SATELLITE")
baseline_map.addLayer(baseline_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "Baseline true color", False)
baseline_map.addLayer(baseline.selfMask(), {"palette": ["blue"]}, "Baseline water")
baseline_map.addLayer(aoi, {"color": "yellow"}, "AOI")
baseline_map

Map(center=[-6.088891396765239, 106.79340900000034], controls=(WidgetControl(options=['position', 'transparent…

## 6. Time-series summary

For every window, how much of the baseline water is still water (`water_ha`) and how much is not (`lost_ha`). Clouds hide part of the baseline water in some windows, so compare windows with `lost_fraction` (lost share of the visible baseline water) rather than raw hectares, and treat windows with a low `clear_fraction` with caution.

In [16]:
time_series = time_series_table(composites, baseline, aoi, NDWI_THRESHOLD, ANALYSIS_SCALE)
time_series

EEException: Too many concurrent aggregations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
time_series.plot(x="window_start", y="lost_fraction", marker="o", ax=axes[0], legend=False, title="Share of visible baseline water lost")
time_series.plot(x="window_start", y=INDEX_NAMES, marker="o", ax=axes[1], title="Mean index over baseline water")
time_series.plot(x="window_start", y="clear_fraction", marker="o", ax=axes[2], legend=False, title="Clear-sky fraction of baseline water")
axes[0].set_ylim(bottom=0)
axes[2].set_ylim(0, 1)
for ax in axes:
    ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()

## 7. Index change over baseline water

Baseline composite compared with the last window, shown only over baseline water.

In [ ]:
change = index_change(baseline_image, end_image).updateMask(baseline)

change_map = geemap.Map()
change_map.centerObject(aoi, 11)
change_map.add_basemap("SATELLITE")
change_map.addLayer(aoi, {"color": "yellow"}, "AOI")
change_map.addLayer(baseline_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "Baseline true color")
change_map.addLayer(end_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "End true color")
change_map.addLayer(change.select("NDVI_change"), {"min": -0.5, "max": 0.5, "palette": ["brown", "white", "green"]}, "NDVI change")
change_map.addLayer(change.select("NDWI_change"), {"min": -0.5, "max": 0.5, "palette": ["red", "white", "blue"]}, "NDWI change")
change_map.addLayer(change.select("NDBI_change"), {"min": -0.5, "max": 0.5, "palette": ["blue", "white", "red"]}, "NDBI change")
change_map

## 8. Baseline water lost by the end

In [ ]:
lost = water_loss(baseline, end_image, NDWI_THRESHOLD)
lost_stats = area_ha(ee.Image.cat([baseline.rename("baseline_water_ha"), lost.rename("water_to_land_ha")]), aoi, ANALYSIS_SCALE).getInfo()
pd.DataFrame([lost_stats])

In [ ]:
water_map = geemap.Map()
water_map.centerObject(aoi, 11)
water_map.add_basemap("SATELLITE")
water_map.addLayer(baseline.selfMask(), {"palette": ["blue"], "opacity": 0.4}, "Baseline water")
water_map.addLayer(lost.selfMask(), {"palette": ["red"]}, "Water to land")
water_map.addLayer(aoi, {"color": "yellow"}, "AOI")
water_map

## 9. Existing coral reefs (Allen Coral Atlas)

Reef habitat mapped from 2018-2021 imagery at 5 m. Coverage is global for shallow tropical reefs; tables are empty when the AOI has no mapped reef.

In [ ]:
coral = coral_atlas()
reef_area = area_ha(coral.select("reef_mask").unmask(0).eq(1).rename("reef_ha"), aoi, ANALYSIS_SCALE).getInfo()
print(f"Mapped reef extent in AOI: {reef_area['reef_ha']:.1f} ha")

geomorphic_table = class_area_table(
    class_area_groups(coral.select("geomorphic"), aoi, ANALYSIS_SCALE).getInfo(), GEOMORPHIC_CLASSES
)
benthic_table = class_area_table(
    class_area_groups(coral.select("benthic"), aoi, ANALYSIS_SCALE).getInfo(), BENTHIC_CLASSES
)
display(geomorphic_table)
display(benthic_table)

In [ ]:
coral_map = geemap.Map()
coral_map.centerObject(aoi, 11)
coral_map.add_basemap("SATELLITE")
# The atlas carries its own class palettes, so no visualization parameters are needed.
coral_map.addLayer(coral.select("geomorphic").selfMask().clip(aoi), {}, "Geomorphic zonation")
coral_map.addLayer(coral.select("benthic").selfMask().clip(aoi), {}, "Benthic habitat")
coral_map.addLayer(coral.select("reef_mask").selfMask().clip(aoi), {}, "Reef extent")
coral_map.addLayer(aoi, {"color": "yellow"}, "AOI")
coral_map

## 10. Potential concern zones

Rule-based flags inside baseline water, comparing the baseline composite with the last window:

- **water_to_land**: baseline water that is no longer water (possible reclamation or fill).
- **builtup_gain**: the part of that lost water whose NDBI rose by more than `NDBI_GAIN_THRESHOLD` (looks built-up rather than mud or sand).
- **near_reef_ha**: hectares of flagged change within `REEF_BUFFER_M` of a mapped reef.

Connected flagged pixels are grouped into zones, and zones smaller than `MIN_ZONE_AREA_HA` are dropped. Zones with more signals are listed first.

Narrow strips along the shoreline are often tide or turbidity differences between composites, not real change.

In [ ]:
flags = concern_flags(
    baseline,
    baseline_image,
    end_image,
    ndwi_threshold=NDWI_THRESHOLD,
    ndbi_gain_threshold=NDBI_GAIN_THRESHOLD,
)
zones = concern_zones(
    flags,
    aoi,
    near_reef=near_reef_mask(REEF_BUFFER_M),
    scale=ANALYSIS_SCALE,
    min_area_ha=MIN_ZONE_AREA_HA,
)
zone_df = zone_table(zones.getInfo())
print(f"{len(zone_df)} concern zone(s) of at least {MIN_ZONE_AREA_HA} ha")
zone_df

In [ ]:
zone_map = geemap.Map()
zone_map.centerObject(aoi, 11)
zone_map.add_basemap("SATELLITE")
zone_map.addLayer(end_image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "End true color", False)
zone_map.addLayer(baseline.selfMask(), {"palette": ["blue"], "opacity": 0.3}, "Baseline water")
zone_map.addLayer(flags.select("water_to_land").selfMask(), {"palette": ["red"]}, "Water to land")
zone_map.addLayer(flags.select("builtup_gain").selfMask(), {"palette": ["magenta"]}, "Built-up gain")
zone_map.addLayer(coral.select("reef_mask").selfMask().clip(aoi), {"palette": ["cyan"]}, "Reef extent")
zone_map.addLayer(ee.Image().paint(zones, 0, 2), {"palette": ["yellow"]}, "Concern zones")
zone_map.addLayer(aoi, {"color": "white"}, "AOI", False)
zone_map

## 11. Not yet implemented

- Confidence score per zone (signal agreement, change size, clear-sky coverage, persistence across windows).
- Candidate businesses/entities near each zone (OpenStreetMap/Overpass, permit and company registries).
- Protected areas (WDPA) and country zoning layers, e.g. Indonesia's RZWP3K.
- Exports to Google Drive and HTML/PDF report generation.